This **set-up cell** - Run it first.

* Qiskit has changed a lot over the years, and some older commands (`Aer`, `BasicAer`, `execute`,
  `IBMQ`) no longer exist. The helper file `qiskit_backend_config.py` (in this folder) provides
  stand-ins for them, so the older code below still works.
* `sys.path.insert(...)` tells Python to look for that helper file in the current folder.
* `BACKEND_NAME = "qasm_simulator"` chooses **where the circuits run**. A *backend* is either a
  simulator on your own computer or a real quantum computer at IBM. The `qasm_simulator` imitates
  a real device: it *measures* the qubits many times ("shots") and counts the results.
* If you have an IBM Quantum account you could set a token and pick a real device, but nothing in
  this notebook needs that.
* The last line prints which backend was created, so you know the cell worked.

In [1]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

Backend ready: LegacyAerSimulator('aer_simulator')


The next code cell **creates a variational form** (also called an **ansatz**; see the next cell for the word)
with Qiskit's `TwoLocal` and prints it.

* A *variational form* is a circuit with **adjustable rotation angles**. An optimiser (a classical
  program) tunes the angles until the circuit's output is as good as possible, like tuning the
  knobs of a radio. This is the basis of variational algorithms (VQE) and quantum machine learning.
* `TwoLocal(2, 'ry', 'cz')` stacks layers of:
  * **`ry`** rotations, one per qubit, each with its own angle $\theta$ ("rotation layer");
  * **`cz`** gates between the qubits, which create entanglement ("entanglement layer").

  "Two-local" means every gate acts on at most two qubits.
* By default there are 3 repetitions of (rotation + entanglement) plus a final rotation layer:
  $4 \times 2 = 8$ angles, $\theta[0] \ldots \theta[7]$, as the printout shows.
* `print(ansatz)` draws it as one box; `print(ansatz.decompose())` would show the individual gates.

*(Newer Qiskit versions recommend the function `n_local(...)` instead of the `TwoLocal` class; it
builds the same circuit.)*

In [2]:
from qiskit.circuit.library import TwoLocal

num_qubits = 2



ansatz = TwoLocal(num_qubits, 'ry', 'cz')
print(ansatz)

     ┌────────────────────────────────────────────────────┐
q_0: ┤0                                                   ├
     │  TwoLocal(θ[0],θ[1],θ[2],θ[3],θ[4],θ[5],θ[6],θ[7]) │
q_1: ┤1                                                   ├
     └────────────────────────────────────────────────────┘


C:\Users\AMITAVA\AppData\Local\Temp\ipykernel_51748\2784367475.py:7: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(num_qubits, 'ry', 'cz')


Ansatz

A term of German origin meaning “initial placement of a tool at a work piece” (Wikipedia), and is used in mathematics to refer to the initial, additional mathematical assumptions made to kick start the problem solving process — but which are later confirmed to be parts of the actual solution as well. Some examples of ansatz in mathematics include:

    The adoption of linear regression model to fit the data points (i.e., linear ansatz)
    The assumption that the solutions to a differential/recurrent equation take an exponential form (exponential ansatz)
    The assumption that the solution to a system of linear equations is expressible in terms of a particular vector (ansatz of particular solution).

Metaphorically speaking, the use of ansatz can be liken to drawing a picture by first establishing a framework, in that if it succeeds, then the framework can be reused as an ansatz later on, but if it doesn’t, then it’s usually discarded and marked as an instance of failure.

The next code cell is a **different topic**: it checks **how accurately a quantum circuit can simulate
the evolution of a physical system in time**.

**Background.** In quantum mechanics a system with energy operator (*Hamiltonian*) $H$ changes in
time as $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$. A quantum computer cannot apply $e^{-iHt}$ in
one step, so it approximates it with a sequence of simple gates: the **Trotter–Suzuki**
approximation.

**Step by step:**

1. `np.random.random((4, 4))` makes a random $4\times4$ matrix; adding its transpose
   (`temp + temp.T`) makes it *symmetric*, so it is a valid Hamiltonian $H$ for 2 qubits.
2. `SparsePauliOp.from_operator(...)` rewrites $H$ as a sum of Pauli terms ($II, XZ, YY, \ldots$),
   the form circuits understand.
3. The starting state is $|00\rangle$; the evolution time is $t = 1$.
4. **Exact answer:** `expm` (matrix exponential from SciPy) computes $e^{-iHt}$ exactly, which we
   apply to the start state.
5. **Circuit answer:** `PauliEvolutionGate` with `SuzukiTrotter(order=2, reps=1)` builds the
   approximate circuit; `evolve` applies it to the start state.
6. `state_fidelity` compares the two states: **1** means identical, **0** completely different.

The printed number is the fidelity. It is usually close to 1, and it changes from run to run because
$H$ is random. Try `reps=3`: more Trotter steps give a better approximation (but a longer circuit).

In [3]:
# Evolution fidelity: how close is a Trotterized evolution exp(-iHt) of a random
# Hamiltonian H to the exact evolution?  (EvolutionFidelity was an example algorithm from the
# Qiskit Aqua tutorials; Aqua's operators / QuantumInstance were removed, so it is written out here.)
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector, state_fidelity
from qiskit.synthesis import SuzukiTrotter

num_qubits = 2
temp = np.random.random((2 ** num_qubits, 2 ** num_qubits))
matrix = temp + temp.T
qubit_op = SparsePauliOp.from_operator(Operator(matrix))

initial_state = Statevector.from_label('0' * qubit_op.num_qubits)   # the "Zero" initial state
evo_time = 1

# exact evolution
exact_state = initial_state.evolve(Operator(expm(-1j * evo_time * matrix)))

# Aqua's suzuki expansion_order=1 == symmetric second-order Suzuki-Trotter circuit
evolution = QuantumCircuit(num_qubits)
evolution.append(PauliEvolutionGate(qubit_op, time=evo_time, synthesis=SuzukiTrotter(order=2, reps=1)), range(num_qubits))
trotter_state = initial_state.evolve(evolution)

result = {'score': state_fidelity(exact_state, trotter_state)}
print(result['score'])

0.9999999999999998


h:\DevelopmentWorkspaces\python-envs\amc-qm\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
h:\DevelopmentWorkspaces\python-envs\amc-qm\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
